In [ ]:

import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from torchvision.transforms import v2
from torch.utils.data import DataLoader
from bayesian_torch.models.dnn_to_bnn import dnn_to_bnn, get_kl_loss
from medmnist import PathMNIST, INFO

#Config
LR            = 0.001
EPOCHS        = 50
BATCH_SIZE    = 128
MILESTONES    = [20, 35, 45]
GAMMA         = 0.5
NUM_MC        = 100
TARGET_ACCS   = [0.99, 0.98, 0.97]
CLASSES_78    = [7, 8]
RESULTS_FILE  = "threshold_results.json"
CKPT_DIR      = "threshold_checkpoints"
FIG_DIR       = "threshold_figures"

BNN_PARAMS = {
    "prior_mu": 0.0,
    "prior_sigma": 1.0,
    "posterior_mu_init": 0.0,
    "posterior_rho_init": -3.0,
    "type": "Reparameterization",
    "moped_enable": False,
    "moped_delta": 0.5,
}

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(FIG_DIR,  exist_ok=True)

device = torch.device(
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available() else "cpu"
)
print(f"Device : {device}")

#Data
transform_eval = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

info        = INFO["pathmnist"]
class_names = list(info["label"].values())
N_CLASSES   = len(class_names)

trainset = PathMNIST(split="train", download=True, size=28, transform=transform_eval)
valset   = PathMNIST(split="val",   download=True, size=28, transform=transform_eval)
testset  = PathMNIST(split="test",  download=True, size=28, transform=transform_eval)

#Balancing taining dataset
BALANCE_SEED          = 42
MAX_OVERSAMPLE_FACTOR = 10

def print_class_distribution(labels, title):
    labels = np.asarray(labels).flatten()
    print(f"\n{title}")
    for c, name in enumerate(class_names):
        n = int((labels == c).sum())
        print(f"  [{c}] {name:<38s} {n:>6d}")
    print(f"  {'TOTAL':<42s} {len(labels):>6d}")


def build_balanced_indices(labels, target_per_class=None, max_oversample_factor=MAX_OVERSAMPLE_FACTOR, seed=BALANCE_SEED):
    labels = np.asarray(labels).flatten()
    rng    = np.random.default_rng(seed)

    counts = np.array([int((labels == c).sum()) for c in range(N_CLASSES)])
    if target_per_class is None:
        target_per_class = int(np.median(counts))

    indices = []
    for c in range(N_CLASSES):
        idx_c = np.where(labels == c)[0]
        n_c   = len(idx_c)

        if n_c >= target_per_class:
            chosen = rng.choice(idx_c, size=target_per_class, replace=False)
        else:
            n_final = min(target_per_class, n_c * max_oversample_factor)
            chosen  = rng.choice(idx_c, size=n_final, replace=True)

        indices.append(chosen)

    indices = np.concatenate(indices)
    rng.shuffle(indices)
    return indices.tolist(), target_per_class


train_labels_raw = trainset.labels.flatten()
print_class_distribution(train_labels_raw, "orignal distribution :")

balanced_idx, TARGET_PER_CLASS = build_balanced_indices(train_labels_raw)
print(f"\nTarget per class (median of original counts) : {TARGET_PER_CLASS}")
print(f"Maximum oversampling factor : x{MAX_OVERSAMPLE_FACTOR}")

trainset_balanced    = torch.utils.data.Subset(trainset, balanced_idx)
train_labels_balanced = train_labels_raw[balanced_idx]
print_class_distribution(train_labels_balanced, "Balanced distribution of the trainset :")

trainloader = DataLoader(trainset_balanced, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
valloader   = DataLoader(valset,            batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
testloader  = DataLoader(testset,           batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"\nTrain (equilibre): {len(trainset_balanced)} | Val: {len(valset)} | Test: {len(testset)}")
print_class_distribution(valset.labels.flatten(),  "Distribution du valset :")
print_class_distribution(testset.labels.flatten(), "Distribution du testset :")

print(f"Classes: {class_names}")

#Model
def build_bnn():
    net = torchvision.models.efficientnet_b0(progress=False)
    net.classifier[1] = nn.Linear(1280, N_CLASSES)
    dnn_to_bnn(net, BNN_PARAMS)
    return net.to(device)

#Training
def train_bnn(tag="bnn"):
    net       = build_bnn()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=MILESTONES, gamma=GAMMA)
    n_train   = len(trainloader.dataset)
    train_losses, val_losses = [], []

    for epoch in range(EPOCHS):
        net.train()
        run_loss = 0.0
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.squeeze(1).to(device)
            optimizer.zero_grad()
            out  = net(inputs)
            kl   = get_kl_loss(net)
            loss = criterion(out, labels) + kl / n_train
            loss.backward()
            optimizer.step()
            run_loss += loss.item()
        train_losses.append(run_loss / len(trainloader))
        scheduler.step()

        net.eval()
        run_val = 0.0
        with torch.no_grad():
            for inputs, labels in valloader:
                inputs, labels = inputs.to(device), labels.squeeze(1).to(device)
                run_val += criterion(net(inputs), labels).item()
        val_losses.append(run_val / len(valloader))

        if (epoch + 1) % 10 == 0 or epoch == EPOCHS - 1:
            print(f"  [{tag}] epoch {epoch+1}/{EPOCHS} — train {train_losses[-1]:.3f} — val {val_losses[-1]:.3f}")

    return net, train_losses, val_losses

#Evaluation
def run_mc(net, loader):
    """Retourne mc_outputs (MC, N, C), labels (N,)."""
    net.eval()
    all_mc, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.squeeze(1).cpu().numpy()
            mc = [F.softmax(net(inputs), dim=-1).cpu().numpy() for _ in range(NUM_MC)]
            all_mc.append(np.stack(mc, axis=0))  
            all_labels.append(labels)
    mc_out = np.concatenate(all_mc,    axis=1)    
    labels = np.concatenate(all_labels, axis=0)  
    return mc_out, labels


def entropy_H(mc_outputs):
    mean_p = mc_outputs.mean(axis=0)               
    return -np.sum(mean_p * np.log(mean_p + 1e-10), axis=-1) 

def find_threshold(H, preds, labels, target_acc, mask=None):

    if mask is not None:
        H_cal    = H[mask]
        preds_cal = preds[mask]
        labels_cal = labels[mask]
    else:
        H_cal    = H
        preds_cal = preds
        labels_cal = labels

    correct_cal = (preds_cal == labels_cal)

    order      = np.argsort(H_cal)
    H_sorted   = H_cal[order]
    cor_sorted = correct_cal[order]

    best_thresh = None
    candidates = np.unique(H_sorted)

    for thresh in candidates:
        accepted = H_cal <= thresh
        if accepted.sum() == 0:
            continue
        acc = correct_cal[accepted].mean()
        if acc >= target_acc:
            best_thresh = float(thresh)

    return best_thresh


def evaluate_with_threshold(mc_outputs, labels, H_thresh):
  
    mean_p  = mc_outputs.mean(axis=0)              # (N, C)
    preds   = mean_p.argmax(axis=-1)               # (N,)
    H       = entropy_H(mc_outputs)                # (N,)

    if H_thresh is None:
        # Seuil impossible : tout le monde répond, target inatteignable
        accepted = np.ones(len(H), dtype=bool)
    else:
        accepted = H <= H_thresh

    correct   = preds == labels
    n_total   = len(labels)
    n_accept  = int(accepted.sum())
    n_abstain = n_total - n_accept
    n_correct = int((correct & accepted).sum())
    n_wrong   = int((~correct & accepted).sum())

    acc_on_accepted = float(n_correct / n_accept) if n_accept > 0 else 0.0
    coverage        = float(n_accept / n_total)

    per_class = {}
    for c in range(N_CLASSES):
        mask_c = labels == c
        per_class[str(c)] = {
            "n_total":   int(mask_c.sum()),
            "n_correct": int((correct & accepted & mask_c).sum()),
            "n_wrong":   int((~correct & accepted & mask_c).sum()),
            "n_abstain": int((~accepted & mask_c).sum()),
        }

    return {
        "H_thresh":         H_thresh,
        "n_total":          n_total,
        "n_accepted":       n_accept,
        "n_abstain":        n_abstain,
        "n_correct":        n_correct,
        "n_wrong":          n_wrong,
        "acc_on_accepted":  acc_on_accepted,
        "coverage":         coverage,
        "per_class":        per_class,
    }


def entropy_scores(mc_outputs):
    mean_p = mc_outputs.mean(axis=0)
    H      = -np.sum(mean_p * np.log(mean_p + 1e-10), axis=-1)
    exp_H  = -np.mean(np.sum(mc_outputs * np.log(mc_outputs + 1e-10), axis=-1), axis=0)
    MI     = H - exp_H
    return H, MI, H - MI


def entropy_per_class(mc_outputs, labels):
    H, MI, aleat = entropy_scores(mc_outputs)
    h_c, mi_c, al_c = [], [], []
    for c in range(N_CLASSES):
        mask = labels == c
        h_c.append(float(H[mask].mean())     if mask.any() else 0.0)
        mi_c.append(float(MI[mask].mean())   if mask.any() else 0.0)
        al_c.append(float(aleat[mask].mean()) if mask.any() else 0.0)
    return {"H": h_c, "MI": mi_c, "H_MI": al_c}


def compute_auce(mc_outputs, labels, n_bins=20):
    H, _, _  = entropy_scores(mc_outputs)
    preds    = mc_outputs.mean(axis=0).argmax(axis=-1)
    correct  = (preds == labels).astype(float)
    order    = np.argsort(H)
    H_sorted = H[order]
    cor_sort = correct[order]
    N        = len(H_sorted)
    bin_size = N // n_bins
    errors, thresholds = [], []
    for i in range(n_bins):
        thresh = H_sorted[min((i + 1) * bin_size - 1, N - 1)]
        err    = 1.0 - cor_sort[:min((i + 1) * bin_size, N)].mean()
        errors.append(float(err))
        thresholds.append(float(thresh / (H_sorted.max() + 1e-10)))
    return float(np.trapezoid(errors, thresholds)), thresholds, errors


def compute_ace(mc_outputs, labels, n_bins=20):
    mean_p  = mc_outputs.mean(axis=0)
    conf    = mean_p.max(axis=-1)
    correct = (mean_p.argmax(axis=-1) == labels).astype(float)
    order   = np.argsort(conf)
    conf_s  = conf[order]
    cor_s   = correct[order]
    N       = len(conf_s)
    bin_size = max(N // n_bins, 1)
    accs, confs = [], []
    for i in range(n_bins):
        sl = slice(i * bin_size, min((i + 1) * bin_size, N))
        if cor_s[sl].size == 0:
            continue
        accs.append(float(cor_s[sl].mean()))
        confs.append(float(conf_s[sl].mean()))
    return float(np.mean(np.abs(np.array(accs) - np.array(confs)))), confs, accs


def compute_uncertain_when_inaccurate(mc_outputs, labels, n_thresholds=50):
    H, _, _ = entropy_scores(mc_outputs)
    preds   = mc_outputs.mean(axis=0).argmax(axis=-1)
    inaccurate = preds != labels
    thresholds = np.linspace(H.min(), H.max(), n_thresholds)
    fracs, vals = [], []
    for t in thresholds:
        uncertain = H >= t
        fracs.append(float(uncertain.mean()))
        vals.append(float((uncertain & inaccurate).sum() / max(inaccurate.sum(), 1)))
    return fracs, vals


def compute_conf_vs_acc(mc_outputs, labels, n_thresholds=50):
    mean_p  = mc_outputs.mean(axis=0)
    conf    = mean_p.max(axis=-1)
    correct = (mean_p.argmax(axis=-1) == labels).astype(float)
    threshs, frac_acc = [], []
    for t in np.linspace(conf.min(), conf.max(), n_thresholds):
        mask = conf >= t
        if mask.sum() == 0:
            continue
        threshs.append(float(t))
        frac_acc.append(float(correct[mask].mean()))
    return threshs, frac_acc


def accuracy_per_class(preds, labels):
    return [float((preds[labels == c] == c).mean()) if (labels == c).any() else 0.0
            for c in range(N_CLASSES)]

#Main
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        all_results = json.load(f)
    print(f"Results loaded from {RESULTS_FILE}")
else:
    all_results = {}


def save_results():
    with open(RESULTS_FILE, "w") as f:
        json.dump(all_results, f, indent=2)


CKPT_PATH = os.path.join(CKPT_DIR, "bnn_nomoped.pth")

if "model_trained" not in all_results:
    print("\nTraining...")
    net, train_losses, val_losses = train_bnn(tag="bnn_nomoped")
    torch.save({
        "model_state_dict": net.state_dict(),
        "train_losses": train_losses,
        "val_losses":   val_losses,
    }, CKPT_PATH)
    all_results["model_trained"]  = True
    all_results["train_losses"]   = train_losses
    all_results["val_losses"]     = val_losses
    save_results()
    print("Checkpoint saved.")
else:
    print("Loading existing model...")
    net = build_bnn()
    ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
    net.load_state_dict(ckpt["model_state_dict"])


# ── 2. Inférence MC sur val et test
print("\nRunning MC inference on validation loader...")
val_mc, val_labels = run_mc(net, valloader)
val_preds = val_mc.mean(axis=0).argmax(axis=-1)
val_H     = entropy_H(val_mc)

print("Running MC inference on test loader...")
test_mc, test_labels = run_mc(net, testloader)
test_preds = test_mc.mean(axis=0).argmax(axis=-1)
test_H     = entropy_H(test_mc)


if "standard_metrics" not in all_results:
    print("\nComputing standard metrics...")
    auce_val, auce_thresh, auce_err = compute_auce(val_mc, val_labels)
    ace_val, ace_confs, ace_accs    = compute_ace(val_mc, val_labels)
    ui_fracs, ui_vals               = compute_uncertain_when_inaccurate(val_mc, val_labels)
    ca_threshs, ca_accs             = compute_conf_vs_acc(val_mc, val_labels)
    ent_by_class                    = entropy_per_class(val_mc, val_labels)
    acc_per_cls                     = accuracy_per_class(test_preds, test_labels)

    all_results["standard_metrics"] = {
        "val_acc":  float((val_preds == val_labels).mean()),
        "test_acc": float((test_preds == test_labels).mean()),
        "entropy_by_class_val": ent_by_class,
        "acc_per_class_test":   acc_per_cls,
        "calibration": {
            "auce": auce_val, "auce_thresholds": auce_thresh, "auce_errors": auce_err,
            "ace":  ace_val,  "ace_confidences": ace_confs,   "ace_accuracies": ace_accs,
            "uncertain_when_inaccurate_fracs": ui_fracs,
            "uncertain_when_inaccurate_vals":  ui_vals,
            "conf_vs_acc_thresholds": ca_threshs,
            "conf_vs_acc_accs":       ca_accs,
        },
    }
    save_results()


mask_78_val = np.isin(val_labels, CLASSES_78)

if "threshold_results" not in all_results:
    all_results["threshold_results"] = {"global": {}, "classes_78": {}}

mask_78_test = np.isin(test_labels, CLASSES_78)

STRATEGY_CONFIGS = [
    ("global",     None,        test_mc,                     test_labels),
    ("classes_78", mask_78_val, test_mc[:, mask_78_test, :], test_labels[mask_78_test]),
]

for strategy, mask_val, eval_mc, eval_labels in STRATEGY_CONFIGS:
    for target in TARGET_ACCS:
        key = f"{int(target*100)}pct"
        if key in all_results["threshold_results"][strategy]:
            print(f"  [{strategy} / {key}] already computed, skip.")
            continue

        print(f"\n  Strategy={strategy} | target={target:.0%}")
        H_thresh = find_threshold(val_H, val_preds, val_labels, target, mask=mask_val)

        if H_thresh is None:
            print(f"    ATTENTION: target {target:.0%} unreachable, threshold = None (all accepted)")
        else:
            cal_H = val_H[mask_val] if mask_val is not None else val_H
            n_accepted_val = int((cal_H <= H_thresh).sum())
            print(f"    H_thresh={H_thresh:.4f} | validation accepted={n_accepted_val}")

        eval_res = evaluate_with_threshold(eval_mc, eval_labels, H_thresh)
        print(f"    test: correct={eval_res['n_correct']} | wrong={eval_res['n_wrong']} | abstain={eval_res['n_abstain']} | acc_accepted={eval_res['acc_on_accepted']:.3f}")

        all_results["threshold_results"][strategy][key] = {
            "target_acc": target,
            "H_thresh":   H_thresh,
            **eval_res,
        }
        save_results()

print(f"\nResults saved to {RESULTS_FILE}")

#Graphs
BLUE   = "#4C72B0"
GREEN  = "#55A868"
RED    = "#C44E52"
ORANGE = "#DD8452"


def plot_entropy_per_class():
    d     = all_results["standard_metrics"]["entropy_by_class_val"]
    x     = np.arange(N_CLASSES)
    bar_w = 0.35
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.bar(x - bar_w/2, d["H"],  bar_w, color=BLUE,  label="H",  alpha=0.9)
    ax.bar(x + bar_w/2, d["MI"], bar_w, color=ORANGE, label="MI", hatch="//", alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("Nats")
    ax.set_title("Entropy / Mutual Information per class (val)")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    path = os.path.join(FIG_DIR, "entropy_per_class.png")
    plt.savefig(path, dpi=130); plt.close()
    print(f"  Save: {path}")


def plot_calibration():
    cal = all_results["standard_metrics"]["calibration"]
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    fig.suptitle("Calibration BayNoMoped PathMNIST (val)", fontsize=13)
    ax_auce, ax_ace, ax_ui, ax_ca = axes[0,0], axes[0,1], axes[1,0], axes[1,1]

    ax_auce.plot(cal["auce_thresholds"], cal["auce_errors"], color=BLUE,
                 label=f"AUCE={cal['auce']:.1%}")
    ax_auce.set_xlabel("Normalized predictive uncertainty"); ax_auce.set_ylabel("Error rate")
    ax_auce.set_title("AUCE"); ax_auce.legend(); ax_auce.grid(alpha=0.3)

    ax_ace.plot(cal["ace_confidences"], cal["ace_accuracies"], color=BLUE,
                label=f"ACE={cal['ace']:.1%}")
    ax_ace.plot([0,1],[0,1],"k--", alpha=0.4, label="Perfect")
    ax_ace.set_xlabel("Confidence"); ax_ace.set_ylabel("Accuracy")
    ax_ace.set_title("ACE"); ax_ace.legend(); ax_ace.grid(alpha=0.3)

    ax_ui.plot(cal["uncertain_when_inaccurate_fracs"], cal["uncertain_when_inaccurate_vals"], color=BLUE)
    ax_ui.set_xlabel("Fraction retained (most uncertain)"); ax_ui.set_ylabel("P(uncertain | inaccurate)")
    ax_ui.set_title("Uncertain when Inaccurate"); ax_ui.grid(alpha=0.3)

    ax_ca.plot(cal["conf_vs_acc_thresholds"], cal["conf_vs_acc_accs"], color=BLUE)
    ax_ca.set_xlabel("Confidence threshold"); ax_ca.set_ylabel("Fraction accurate predictions")
    ax_ca.set_title("Confidence vs Accuracy"); ax_ca.grid(alpha=0.3)

    plt.tight_layout()
    path = os.path.join(FIG_DIR, "calibration.png")
    plt.savefig(path, dpi=130); plt.close()
    print(f"  Save: {path}")


def plot_accuracy_per_class():
    accs = all_results["standard_metrics"]["acc_per_class_test"]
    x    = np.arange(N_CLASSES)
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.bar(x, [a * 100 for a in accs], color=BLUE, alpha=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("Accuracy (%)")
    ax.set_title("Accuracy per class (test, no abstention)")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    path = os.path.join(FIG_DIR, "accuracy_per_class.png")
    plt.savefig(path, dpi=130); plt.close()
    print(f"  Save: {path}")


def plot_abstention_histogram(strategy):
    data    = all_results["threshold_results"][strategy]
    targets = ["99pct", "98pct", "97pct"]
    labels_x = ["99%", "98%", "97%"]

    n_correct = [data[t]["n_correct"] for t in targets]
    n_wrong   = [data[t]["n_wrong"]   for t in targets]
    n_abstain = [data[t]["n_abstain"] for t in targets]
    n_total   = data[targets[0]]["n_total"]

    x     = np.arange(len(targets))
    bar_w = 0.5

    fig, ax = plt.subplots(figsize=(8, 6))
    bars_c = ax.bar(x, n_correct, bar_w, label="Correct",   color=GREEN,  alpha=0.9)
    bars_w = ax.bar(x, n_wrong,   bar_w, label="Incorrect", color=RED,    alpha=0.9, bottom=n_correct)
    bars_a = ax.bar(x, n_abstain, bar_w, label="Abstain (unknown)", color=ORANGE, alpha=0.9,
                    bottom=[c + w for c, w in zip(n_correct, n_wrong)])

    # Annotations
    for i, (c, w, a) in enumerate(zip(n_correct, n_wrong, n_abstain)):
        ax.text(i, c / 2,         f"{c}",   ha="center", va="center", fontsize=9, color="white", fontweight="bold")
        ax.text(i, c + w / 2,     f"{w}",   ha="center", va="center", fontsize=9, color="white", fontweight="bold")
        ax.text(i, c + w + a / 2, f"{a}",   ha="center", va="center", fontsize=9, color="white", fontweight="bold")
        thresh = data[targets[i]]["H_thresh"]
        t_str  = f"H≤{thresh:.3f}" if thresh is not None else "H≤∞"
        ax.text(i, n_total + n_total * 0.01, t_str, ha="center", va="bottom", fontsize=8, color="gray")

    title_suffix = "global valloader" if strategy == "global" else "classes 7 & 8 valloader"
    ax.set_xticks(x)
    ax.set_xticklabels([f"Target {l}" for l in labels_x])
    ax.set_ylabel("Number of samples (testloader)")
    ax.set_title(f"Model responses on testloader\n(threshold calibrated on {title_suffix})")
    ax.legend()
    ax.set_ylim(0, n_total * 1.08)
    ax.grid(axis="y", alpha=0.3)
    ax.axhline(n_total, color="black", linestyle="--", alpha=0.4, linewidth=1)
    ax.text(len(targets) - 0.5, n_total, f"Total={n_total}", va="bottom", fontsize=8, color="gray")

    plt.tight_layout()
    path = os.path.join(FIG_DIR, f"abstention_{strategy}.png")
    plt.savefig(path, dpi=130); plt.close()
    print(f"  Save: {path}")


def plot_abstention_per_class(strategy, target_key):
    data  = all_results["threshold_results"][strategy][target_key]
    x     = np.arange(N_CLASSES)
    bar_w = 0.6

    n_c = [data["per_class"][str(c)]["n_correct"] for c in range(N_CLASSES)]
    n_w = [data["per_class"][str(c)]["n_wrong"]   for c in range(N_CLASSES)]
    n_a = [data["per_class"][str(c)]["n_abstain"] for c in range(N_CLASSES)]

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.bar(x, n_c, bar_w, label="Correct",       color=GREEN,  alpha=0.9)
    ax.bar(x, n_w, bar_w, label="Incorrect",      color=RED,    alpha=0.9, bottom=n_c)
    ax.bar(x, n_a, bar_w, label="Abstain (unknown)",  color=ORANGE, alpha=0.9,
           bottom=[c + w for c, w in zip(n_c, n_w)])

    thresh = data["H_thresh"]
    t_str  = f"H ≤ {thresh:.3f}" if thresh is not None else "H ≤ ∞"
    suffix = "global" if strategy == "global" else "classes 7&8"
    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("Number of samples (testloader)")
    ax.set_title(f"Responses per class — target {target_key} calibrated on {suffix}\n(threshold {t_str})")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    path = os.path.join(FIG_DIR, f"abstention_per_class_{strategy}_{target_key}.png")
    plt.savefig(path, dpi=130); plt.close()
    print(f"  Save: {path}")


print("\nGraphs generation...")
plot_entropy_per_class()
plot_calibration()
plot_accuracy_per_class()
plot_abstention_histogram("global")
plot_abstention_histogram("classes_78")

for strategy in ["global", "classes_78"]:
    for target_key in ["99pct", "98pct", "97pct"]:
        plot_abstention_per_class(strategy, target_key)

print("\nDone.")

Device : cuda

Distribution originale du trainset :
  [0] adipose                                  9366
  [1] background                               9509
  [2] debris                                  10360
  [3] lymphocytes                             10401
  [4] mucus                                    8006
  [5] smooth muscle                           12182
  [6] normal colon mucosa                      7886
  [7] cancer-associated stroma                 9401
  [8] colorectal adenocarcinoma epithelium    12885
  TOTAL                                       89996

Cible par classe (mediane des comptes originaux) : 9509
Facteur max de sur-echantillonnage : x3

Distribution du trainset apres equilibrage :
  [0] adipose                                  9509
  [1] background                               9509
  [2] debris                                   9509
  [3] lymphocytes                              9509
  [4] mucus                                    9509
  [5] smooth muscle      

In [1]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from torchvision.transforms import v2
from torch.utils.data import DataLoader
from bayesian_torch.models.dnn_to_bnn import dnn_to_bnn, get_kl_loss
from medmnist import PathMNIST, INFO

# ──────────────────────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────────────────────

LR            = 0.001
EPOCHS        = 50
BATCH_SIZE    = 128
MILESTONES    = [20, 35, 45]
GAMMA         = 0.5
NUM_MC        = 100
TARGET_ACCS   = [0.99, 0.98, 0.97]
CLASSES_78    = [7, 8]
RESULTS_FILE  = "threshold_results.json"
CKPT_DIR      = "threshold_checkpoints"
FIG_DIR       = "threshold_figures"

BNN_PARAMS = {
    "prior_mu": 0.0,
    "prior_sigma": 1.0,
    "posterior_mu_init": 0.0,
    "posterior_rho_init": -3.0,
    "type": "Reparameterization",
    "moped_enable": False,
    "moped_delta": 0.5,
}

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(FIG_DIR,  exist_ok=True)

device = torch.device(
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available() else "cpu"
)
print(f"Device : {device}")

transform_eval = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

info        = INFO["pathmnist"]
class_names = list(info["label"].values())
N_CLASSES   = len(class_names)

trainset = PathMNIST(split="train", download=True, size=28, transform=transform_eval)
valset   = PathMNIST(split="val",   download=True, size=28, transform=transform_eval)
testset  = PathMNIST(split="test",  download=True, size=28, transform=transform_eval)

print(trainset)
print(valset)
print(testset)

trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
valloader   = DataLoader(valset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
testloader  = DataLoader(testset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
print(len(trainset), len(valset), len(testset))




Device : cuda
Dataset PathMNIST of size 28 (pathmnist)
    Number of datapoints: 89996
    Root location: /home/a992251/.medmnist
    Split: train
    Task: multi-class
    Number of channels: 3
    Meaning of labels: {'0': 'adipose', '1': 'background', '2': 'debris', '3': 'lymphocytes', '4': 'mucus', '5': 'smooth muscle', '6': 'normal colon mucosa', '7': 'cancer-associated stroma', '8': 'colorectal adenocarcinoma epithelium'}
    Number of samples: {'train': 89996, 'val': 10004, 'test': 7180}
    Description: The PathMNIST is based on a prior study for predicting survival from colorectal cancer histology slides, providing a dataset (NCT-CRC-HE-100K) of 100,000 non-overlapping image patches from hematoxylin & eosin stained histological images, and a test dataset (CRC-VAL-HE-7K) of 7,180 image patches from a different clinical center. The dataset is comprised of 9 types of tissues, resulting in a multi-class classification task. We resize the source images of 3×224×224 into 3×28×28, and